TARDIS-EDA (Étapes 1 & 2)

Étape 1- Data Exploration &Cleaning

# load dataset

In [1]:
import pandas as pd

df = pd.read_csv("dataset.csv", sep=";")

# scan for errors

In [72]:
nul = df.isnull().sum()
isdupe = df.duplicated().sum()
print(nul)
print("num of duplicates:", isdupe)

Date                                                                               541
Service                                                                            552
Departure station                                                                  542
Arrival station                                                                    547
Average journey time                                                               830
Number of scheduled trains                                                         822
Number of cancelled trains                                                         828
Cancellation comments                                                            10840
Number of trains delayed at departure                                              826
Average delay of late trains at departure                                          810
Average delay of all trains at departure                                           823
Departure delay comments                   

On compte les valeurs manquantes et les doublons avant de toucher à quoi que ce soit, pour savoir à quoi s'attendre ensuite.

# clean duplicates


In [73]:
df.drop_duplicates(inplace=True)
df

,Date,Service,Departure station,Arrival station,Average journey time,Number of scheduled trains,Number of cancelled trains,Cancellation comments,Number of trains delayed at departure,Average delay of late trains at departure,...,Number of trains delayed > 15min,Average delay of trains > 15min (if competing with flights),Number of trains delayed > 30min,Number of trains delayed > 60min,Pct delay due to external causes,Pct delay due to infrastructure,Pct delay due to traffic management,Pct delay due to rolling stock,Pct delay due to station management and equipment reuse,"Pct delay due to passenger handling (crowding, disabled persons, connections)"
0,2018-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141.000000,NaN,5.0,NaN,289.0,11.247809,...,110.0,346.474287,44.0,8.0,36.134454,31.092437,10.924370,15.966387,5.042017,75.915730
1,2018-01,National,LA ROCHELLE VILLE,PARIS MONTPARNASSE,165.000000,222.000000,NaN,NaN,8.0,2.875000,...,22.0,5.696096,5.0,NaN,15.384615,30.769231,38.461538,11.538462,3.846154,0.000000
2,2018-01,National,PARIS MONTPARNASSE,QUIMPER,220.000000,248.000000,1.0,NaN,37.0,9.501351,...,26.0,7.548387,17.0,7.0,26.923077,38.461538,NaN,19.230769,0.000000,0.000000
3,2018j01,National,PARIS MONTPARNASSE,ST MALO,156.000000,102.000000,0.0,NaN,12.0,19.912500,...,8.0,6.724757,6.0,4.0,23.076923,218.650888,7.692308,15.384615,7.692308,NaN
4,2018-01,National,PARIS MONTPARNASSE,ST PIERRE DES CORPS,61.000000,391.000000,2.0,NaN,61.0,NaN,...,17.0,3.346487,6.0,0.0,21.212121,42.424242,9.090909,21.212121,6.060606,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10835,2020-04,National,PARIS EST,STRASBOURG,NaN,40.000000,35.0,NaN,5.0,1.253333,...,3.0,NaN,3.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
10836,2020-05,National,NaN,LYON PART DIEU,115.000000,2681.206158,14.0,NaN,46.0,6.258333,...,5.0,47.820000,3.0,1.0,0.000000,50.000000,0.000000,0.000000,50.000000,0.000000
10837,2021-03,National,PARIS LYON,VALENCE ALIXAN TGV,130.000000,178.000000,10.0,NaN,5.0,2.946667,...,4.0,66.254167,4.0,2.0,25.000000,25.000000,198.761036,25.000000,0.000000,25.000000
10838,2019-07,National,MARNE LA VALLEE,MARSEILLE ST CHARLES,217.000000,268.000000,NaN,NaN,238.0,10.391667,...,66.0,50.361364,34.0,22.0,24.615385,15.384615,15.384615,24.615385,10.769231,9.230769


On supprime les lignes strictement identiques (copies exactes).

# remove null entries

In [74]:
df.fillna(0, inplace=True)

⚠️ Limite connue : remplacer les valeurs manquantes par `0` peut fausser les moyennes (un retard "non mesuré" devient un retard "nul"). On le garde ici pour rester simple, mais c'est un choix à assumer et documenter plutôt qu'une vérité absolue.

# nettoyer les valeurs sales (gares, nombres)

In [ ]:
df["Departure station"] = df["Departure station"].str.strip().str.upper()
df["Arrival station"] = df["Arrival station"].str.strip().str.upper()


def to_num(series):
    """Convertit une colonne text sale (virgules, espaces, 'min', '%') en nombres."""
    cleaned = (
        series.astype(str)
        .str.replace(",", ".", regex=False)
        .str.replace("min", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
    )
    return pd.to_numeric(cleaned, errors="coerce")

# transform to correct types

In [ ]:
exclude_columns = ["Date", "Service", "Departure station", "Arrival station"]
columns_to_convert = [col for col in df.columns if col not in exclude_columns]

for col in columns_to_convert:
    df[col] = pd.to_numeric(df[col], errors="coerce")

On applique `to_num` (au lieu d'un simple `pd.to_numeric`) pour absorber les formats sales avant la conversion, comme demandé dans le memo.

In [ ]:
for col in columns_to_convert:
    df[col] = to_num(df[col])

# feature ingineering

In [ ]:
import numpy as np

bonus_values = np.where(
    df["Number of scheduled trains"] > 0,
    (df["Number of cancelled trains"] / df["Number of scheduled trains"]) * 100,
    np.nan,
)
df.insert(loc=8, column="Percentage of cancel", value=bonus_values)
df

⚠️ Correction : un taux d'annulation, c'est (annulés / prévus) × 100 — pas une multiplication. L'ancienne formule (`cancelled * scheduled / 100`) donnait des valeurs fausses dès que le nombre de trains prévus changeait.

# correct date

Certaines dates ont un séparateur corrompu (ex: `2018j01` au lieu de `2018-01`) : on le corrige d'abord. Les dates dont un *chiffre* est corrompu (ex: `2018-r1`) restent irrécupérables sans info externe et deviennent `NaT` — c'est attendu.

In [ ]:
def fix_date_separator(value):
    s = str(value)
    if len(s) == 7:
        return s[:4] + "-" + s[5:]
    return s


df["Date"] = df["Date"].apply(fix_date_separator)

In [78]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

#  finalize dataset

# feature engineering (temporel, trajet, ponctualité)

In [ ]:
df["Month"] = df["Date"].dt.month

season_map = {
    12: "Winter",
    1: "Winter",
    2: "Winter",
    3: "Spring",
    4: "Spring",
    5: "Spring",
    6: "Summer",
    7: "Summer",
    8: "Summer",
    9: "Autumn",
    10: "Autumn",
    11: "Autumn",
}
df["Season"] = df["Month"].map(season_map)

df["Route"] = df["Departure station"] + " -> " + df["Arrival station"]

df["Punctuality rate"] = np.where(
    df["Number of scheduled trains"] > 0,
    (
        1
        - (
            df["Number of trains delayed at departure"]
            / df["Number of scheduled trains"]
        )
    )
    * 100,
    np.nan,
)

`Month` et `Season` viennent de `Date` (d'où la nécessité de corriger les dates avant). `Route` combine gare de départ et d'arrivée. `Punctuality rate` est le complément du taux de trains en retard au départ, en pourcentage.

In [79]:
df.to_csv("cleaned_dataset.csv")

Étape 2-Data Visualisation & Analysis

## 1. Statistiques descriptives globales

On analyse les indicateurs cles du dataset apres nettoyage :
- **Retards moyens** (a l'arrivee et au depart)
- **Taux de ponctualite** et **taux d'annulation**
- **Volumes de trains prevus**

In [ ]:
key_metrics = [
    "Average delay of all trains at arrival",
    "Average delay of all trains at departure",
    "Average delay of late trains at arrival",
    "Punctuality rate",
    "Percentage of cancel",
    "Number of scheduled trains",
]

df[key_metrics].describe().round(2)

In [ ]:
# Comparaison moyenne vs mediane pour reperer les asymetries
stats_comparison = pd.DataFrame({
    "Moyenne": df[key_metrics].mean().round(2),
    "Mediane": df[key_metrics].median().round(2),
    "Ecart (Moy - Med)": (df[key_metrics].mean() - df[key_metrics].median()).round(2),
})
stats_comparison

### Interpretation — Statistiques descriptives
- **Retard moyen a l'arrivee :** La moyenne est de **5.93 min**, alors que la mediane est a **5.26 min**. Cet ecart positif montre une distribution asymetrique vers la droite (quelques retards extremes tirent la moyenne vers le haut).
- **Taux de ponctualite :** La mediane est a **77.49%**, mais la moyenne tombe a **68.44%**. Cela met en evidence l'impact de mois de crise (greves, intemperies) qui degradent fortement la moyenne.
- **Annulations :** Le taux d'annulation median est tres bas (**0.66%**), mais la moyenne grimpe a **3.79%** en raison de pics ponctuels de perturbations.

## 2. Distribution des retards

On analyse la distribution de notre variable cible (**retard moyen a l'arrivee**) :
- Histogramme avec courbe de densite (KDE)
- Filtrage des percentiles 1% - 99% pour eliminer les valeurs extremes qui ecrasent le graphique
- Boxplot montrant les valeurs aberrantes (outliers)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Filtrage entre le 1er et le 99eme percentile pour une visualisation nette
p_low = df["Average delay of all trains at arrival"].quantile(0.01)
p_high = df["Average delay of all trains at arrival"].quantile(0.99)
filtered_delays = df[
    (df["Average delay of all trains at arrival"] >= p_low)
    & (df["Average delay of all trains at arrival"] <= p_high)
]["Average delay of all trains at arrival"]

plt.figure(figsize=(9, 4.5))
sns.histplot(filtered_delays, kde=True, bins=40, color="#2b5c8f")
plt.axvline(filtered_delays.mean(), color="red", linestyle="--", label=f"Moyenne ({filtered_delays.mean():.2f} min)")
plt.axvline(filtered_delays.median(), color="green", linestyle="-", label=f"Mediane ({filtered_delays.median():.2f} min)")
plt.title("Distribution du retard moyen a l'arrivee (percentiles 1% - 99%)")
plt.xlabel("Retard moyen a l'arrivee (minutes)")
plt.ylabel("Nombre d'observations")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot pour identifier la dispersion et les valeurs extremes
plt.figure(figsize=(9, 2.5))
sns.boxplot(x=df["Average delay of all trains at arrival"], color="#729fcf", showfliers=True)
plt.title("Dispersion globale et valeurs extremes (Boxplot)")
plt.xlabel("Retard moyen a l'arrivee (minutes)")
plt.tight_layout()
plt.show()

### Interpretation — Distribution des retards
- **Asymetrie a droite (right-skewed) :** La grande majorite des trajets a un retard moyen concentre entre **2 et 7 minutes**. La queue de distribution s'etire nettement vers les retards eleves.
- **Presence d'outliers :** Le boxplot met en evidence de fortes valeurs extremes allant au-dela de 30 a 60 minutes de retard moyen mensuel, ainsi que quelques valeurs negatives issues de donnees corrompues dans le brut. Il sera utile d'en tenir compte lors de l'entrainement des modeles.

## 3. Retards par gares et par trajets

On compare les performances entre gares et entre liaisons :
- Top 10 des gares de depart avec le plus fort retard moyen a l arrivée (au moins 30 observations pour la fiabilite)
- Top 10 des trajets (Route) les plus retardes (au moins 20 observations)

In [ ]:
# Top 10 des gares de depart les plus en retard
top_stations = (
    df.groupby("Departure station")["Average delay of all trains at arrival"]
    .agg(["mean", "count"])
    .query("count >= 30")
    .sort_values("mean", ascending=False)
    .head(10)
)

plt.figure(figsize=(9, 4.5))
sns.barplot(x=top_stations["mean"], y=top_stations.index, hue=top_stations.index, palette="Reds_r")
plt.title("Top 10 des gares de depart avec le plus fort retard moyen (min. 30 obs)")
plt.xlabel("Retard moyen a l'arrivee (minutes)")
plt.ylabel("Gare de depart")
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 des trajets les plus retardes
top_routes = (
    df.groupby("Route")["Average delay of all trains at arrival"]
    .agg(["mean", "count"])
    .query("count >= 20")
    .sort_values("mean", ascending=False)
    .head(10)
)

plt.figure(figsize=(9, 4.5))
sns.barplot(x=top_routes["mean"], y=top_routes.index, hue=top_routes.index, palette="flare")
plt.title("Top 10 des trajets les plus en retard (min. 20 obs)")
plt.xlabel("Retard moyen a l'arrivee (minutes)")
plt.ylabel("Trajet")
plt.tight_layout()
plt.show()

### Interpretation — Gares et trajets
- **Impact des liaisons internationales :** Les trajets en provenance d Italie (ITALIE -> PARIS LYON, ~13.7 min) ou d Espagne (BARCELONA -> PARIS LYON, ~10.5 min) dominent le classement des retards. Les changements d infrastructure et d operateurs accentuent les aleas.
- **Trajets transversaux longue distance :** Les lignes province-province (MARSEILLE ST CHARLES -> LILLE, LYON PART DIEU -> LILLE) cumulent de forts retards (> 10 min). Plus un trajet est long et traverse de nœuds ferroviaires denses, plus les retards se propagent.

In [ ]:
# TODO: visualisations (distributions, comparaisons, corrélations)

TODO: interprétations et insights